# Recollida de dades

Aquest notebook genera la mostra inicial del treball: 1000 colors RGB aleatoris, el seu chroma i les imatges PNG que despres passarem als models.

La logica important esta a `scripts/utils.py`. Aqui intento deixar nomes la configuracio i les crides principals, perque si no el notebook es fa impossible de seguir.

## Configuracio

Fixem els parametres de l'experiment. La seed es important per poder regenerar exactament la mateixa mostra en un altre ordinador.

In [ ]:
from pathlib import Path
import sys

N_COLORS = 1000
SEED = 23
IMAGE_SIZE = 100
NEAR_DISTANCE = 30

# Canvia a True nomes si vols tornar a generar la mostra des de zero.
REGENERATE_SAMPLE = False

# Casella de seguretat: si es posa a True, esborra CSVs, imatges PNG i logs generats.
DELETE_EXISTING_SAMPLE = False

base_dir = Path("recollida-dades") if Path("recollida-dades").exists() else Path(".")
csv_dir = base_dir / "csv"
images_dir = base_dir / "images"
tmp_dir = base_dir / "tmp"
logs_dir = base_dir / "logs"
scripts_dir = base_dir / "scripts"

for folder in [csv_dir, images_dir, tmp_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print("Carpeta base:", base_dir)


## Carrega de scripts

Importem les funcions del script. Si alguna no existeix, parem aqui, perque probablement el notebook no esta trobant be `scripts/utils.py`.

In [ ]:
import importlib
import pandas as pd
from IPython.display import Image as DisplayImage, display

import utils
importlib.reload(utils)

build_final_sample = utils.build_final_sample
collect_model_outputs = utils.collect_model_outputs
delete_sample_outputs = utils.delete_sample_outputs
generate_images_from_sample = utils.generate_images_from_sample
generate_unique_colors = utils.generate_unique_colors
save_csv = utils.save_csv
save_rgb_sample_grid = utils.save_rgb_sample_grid
save_rgb_sample_map = utils.save_rgb_sample_map
write_log = utils.write_log

required_functions = [
    "generate_unique_colors",
    "generate_images_from_sample",
    "save_rgb_sample_grid",
    "save_rgb_sample_map",
    "delete_sample_outputs",
    "save_csv",
    "write_log",
]
missing = [name for name in required_functions if name not in globals()]

if missing:
    raise RuntimeError(f"Error: scripts no cargados correctamente: {missing}")

if DELETE_EXISTING_SAMPLE:
    removed = delete_sample_outputs(csv_dir=csv_dir, images_dir=images_dir, logs_dir=logs_dir)
    print(f"Fitxers esborrats: {len(removed)}")

write_log("Notebook de recollida inicialitzat", logs_dir / "pipeline.log")


## Generacio o carrega de la mostra de colors

Si `input_image_sample.csv` ja existeix, el carreguem i no el tornem a generar. Aixi evitem canviar la mostra sense voler.

Per regenerar-ho tot, cal posar `DELETE_EXISTING_SAMPLE = True` i `REGENERATE_SAMPLE = True` a la configuracio. Ho deixo aixi una mica explicit per no liar-la per accident.


In [ ]:
input_path = csv_dir / "input_image_sample.csv"

if input_path.exists() and not REGENERATE_SAMPLE:
    input_sample = pd.read_csv(input_path)
    print(f"Mostra carregada: {input_path}")
else:
    input_sample = generate_unique_colors(n=N_COLORS, seed=SEED)
    save_csv(input_sample, input_path)
    write_log(f"Mostra generada: {input_path}", logs_dir / "pipeline.log")
    print(f"Mostra generada: {input_path}")

input_sample.head()


## Mapa de color i comprovacio rapida de biaix

Generem dues visualitzacions:

- Una graella amb els 1000 colors exactes de la mostra.
- Un mapa de diagnosi amb histogrames RGB, projeccions R-G/R-B/G-B i hue-saturation.

La graella serveix per veure els colors reals. El mapa de diagnosi serveix millor per comprovar si la mostra sembla uniformement repartida: els histogrames haurien de quedar bastant plans i els punts dels plans RGB haurien d'omplir els quadrats sense patrons massa evidents.

Els punts es pinten amb el seu color real, sense contorns blancs o negres, per no confondre la visualitzacio.


In [ ]:
color_grid_path = tmp_dir / "rgb_sample_grid.png"
color_map_path = tmp_dir / "rgb_sample_map.png"

# Les visualitzacions son barates de generar, aixi que les fem sempre a partir del CSV actual.
# D'aquesta manera no ens quedem mirant una imatge antiga del notebook.
save_rgb_sample_grid(input_sample, color_grid_path)
save_rgb_sample_map(input_sample, color_map_path)
write_log(f"Visualitzacions RGB generades a tmp: {color_grid_path}, {color_map_path}", logs_dir / "pipeline.log")

print(f"Graella generada: {color_grid_path}")
print(f"Mapa de diagnosi generat: {color_map_path}")

print("Graella de les 1000 mostres:")
display(DisplayImage(filename=str(color_grid_path)))

print("Mapa de diagnosi de distribucio:")
display(DisplayImage(filename=str(color_map_path)))

input_sample[["r", "g", "b", "chroma"]].describe()


## Generacio de les imatges

Les imatges individuals tambe es reutilitzen si ja existeixen. Si falten o si activem `REGENERATE_SAMPLE`, es tornen a crear.

Cada imatge es de 100x100 pixels: 60% color objectiu, 20% color proper i 20% color aleatori. Les guardem en PNG per no modificar el color amb compressio JPEG.


In [ ]:
expected_images = {name for name in input_sample["image_name"]}
existing_images = {path.name for path in images_dir.glob("*.png")}
missing_images = expected_images - existing_images

if missing_images or REGENERATE_SAMPLE:
    image_paths = generate_images_from_sample(
        input_sample,
        output_dir=images_dir,
        size=IMAGE_SIZE,
        near_distance=NEAR_DISTANCE,
        seed=SEED,
    )
    write_log(f"Imatges generades: {len(image_paths)}", logs_dir / "pipeline.log")
    print(f"Imatges generades: {len(image_paths)}")
else:
    image_paths = sorted(images_dir.glob("*.png"))
    print(f"Imatges ja existents: {len(image_paths)}")

len(image_paths), image_paths[:3]


## Consulta als models

Aquesta part queda preparada pero no l'executo automaticament. Si la llancem sense voler, gastarem crides d'API.

Quan tinguem la clau configurada, la idea es generar `outputmodel_image_sample.csv` amb una fila per imatge i model.

In [ ]:
# from openai import OpenAI
#
# client = OpenAI()
# models = ["gpt-4o", "gpt-4o-mini"]
# model_outputs = collect_model_outputs(
#     client=client,
#     image_paths=image_paths,
#     models=models,
#     temperature=0.2,
# )
#
# output_path = csv_dir / "outputmodel_image_sample.csv"
# save_csv(model_outputs, output_path)
# write_log(f"Resultats de models guardats: {output_path}", logs_dir / "pipeline.log")

## Dataset final per analisi

Quan existeixi el CSV de sortida dels models, unim l'entrada i la sortida i calculem l'error cromatic en RGB. Aquest fitxer sera el que carregarem a R.

In [ ]:
# output_path = csv_dir / "outputmodel_image_sample.csv"
# final_path = csv_dir / "sample-colors.csv"
#
# if output_path.exists():
#     model_outputs = pd.read_csv(output_path)
#     final_sample = build_final_sample(input_sample, model_outputs)
#     save_csv(final_sample, final_path)
#     final_sample.head()
# else:
#     print("Encara no existeix outputmodel_image_sample.csv")

## Resultats generats

Comprovem que tenim el CSV d'entrada i les imatges. Si aqui surt 1000, la part base de recollida ja esta feta.

In [ ]:
print("CSV generats:", sorted(path.name for path in csv_dir.glob("*.csv")))
print("Nombre d'imatges:", len(list(images_dir.glob("*.png"))))
print("Fitxers temporals:", sorted(path.name for path in tmp_dir.glob("*.png")))
print("Logs:", sorted(path.name for path in logs_dir.iterdir()))